# A2 – Rentabilidad y comportamiento de compra

El enunciado pide cuatro cosas para A2:

1. Qué segmentos de clientes, categorías de producto y canales de venta aportan más y menos al **margen bruto** total.
2. El valor de un cliente a lo largo del tiempo (CLV) por segmento, y cuáles segmentos están en declive.
3. La estacionalidad de las ventas: picos, valles y tendencias por categoría y por canal.
4. Si los descuentos generan volumen suficiente para compensar el margen que se pierde.

Usamos los datos limpios de A1 (`cargar_limpio`). Los montos están en la moneda de los datos; el enunciado no dice cuál es.

Cómo calculamos cada cosa, y qué está pendiente de confirmar con el profesor, está en `DECISIONES.md`, sección A2.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from src.carga import cargar_limpio, RAIZ

d = cargar_limpio()
cli = d["clientes"]
trx = d["transacciones"].merge(cli[["cliente_id", "segmento"]], on="cliente_id")
trx["anio"] = trx.fecha.dt.year
FIG = RAIZ / "figures"

# Colores (paleta de referencia de la guía de visualización)
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8a8984"
TEXTO, TEXTO_2, FONDO = "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": "#d6d5d0", "axes.labelcolor": TEXTO_2, "xtick.color": TEXTO_2, "ytick.color": TEXTO_2,
    "text.color": TEXTO, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#ecebe7", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

def miles(x):
    # 1.234 en vez de 1,234
    return f"{x:,.0f}".replace(",", ".")

## 1. Margen bruto y margen neto

- **Margen bruto:** lo que se vendió menos lo que costaron los productos. Es lo que pide el enunciado.
- **Margen neto:** el margen bruto después de restar las devoluciones. Si una venta se devuelve, se reembolsa completa. Si el producto vuelve en buen estado, se revende y solo se pierde el margen de esa venta. Si es defectuoso o un fraude, se pierde también el costo del producto.

También calculamos cuánto margen se regaló en descuentos.

In [ ]:
ventas = trx.monto_neto.sum()
bruto = trx.margen_bruto.sum()
neto = trx.margen_neto.sum()
regalado = trx.costo_descuento.sum()

devueltas = trx[trx.devuelta]
perdido_revendible = devueltas.loc[~devueltas.motivo_devolucion.isin(["Producto_defectuoso", "Fraude_reportado"]), "margen_bruto"].sum()
perdido_costo = bruto - neto - perdido_revendible

print(f"Ventas: {ventas:,.0f}")
print(f"Margen bruto: {bruto:,.0f} ({bruto / ventas:.1%} de las ventas)")
print(f"Margen neto (después de devoluciones): {neto:,.0f} ({neto / ventas:.1%} de las ventas)")
print(f"Las devoluciones se llevan {bruto - neto:,.0f}, el {(bruto - neto) / bruto:.1%} del margen bruto:")
print(f"   margen de ventas devueltas que se revenden: {perdido_revendible:,.0f}")
print(f"   productos defectuosos o fraude (margen + costo): {perdido_costo:,.0f}")
print(f"Descuentos regalados: {regalado:,.0f}. Sin descuentos, el margen bruto habría sido {bruto + regalado:,.0f}: "
      f"los descuentos se llevan el {regalado / (bruto + regalado):.1%}")
print(f"% de ventas devueltas (en monto): {devueltas.monto_neto.sum() / ventas:.1%}")

## 2. Quién aporta el margen

Para cada segmento, categoría y canal mostramos qué parte de las ventas, del margen bruto y del margen neto le corresponde. Si una categoría aporta mucho más a las ventas que al margen, vende mucho pero gana poco.

In [ ]:
def aporte(col):
    g = trx.groupby(col).agg(ventas=("monto_neto", "sum"), margen_bruto=("margen_bruto", "sum"),
                             margen_neto=("margen_neto", "sum"), devuelto=("monto_neto", lambda s: s[trx.loc[s.index, "devuelta"]].sum()))
    t = pd.DataFrame({
        "% de las ventas": 100 * g.ventas / g.ventas.sum(),
        "% del margen bruto": 100 * g.margen_bruto / g.margen_bruto.sum(),
        "% del margen neto": 100 * g.margen_neto / g.margen_neto.sum(),
        "margen bruto %": 100 * g.margen_bruto / g.ventas,
        "margen neto %": 100 * g.margen_neto / g.ventas,
        "% devuelto": 100 * g.devuelto / g.ventas,
        "margen bruto": g.margen_bruto,
    })
    return t.sort_values("% del margen bruto", ascending=False).round(1)

tablas = {col: aporte(col) for col in ["segmento", "categoria", "canal"]}
for col, t in tablas.items():
    print(f"\n=== Por {col}")
    print(t.to_string())
    assert abs(t["margen bruto"].sum() - bruto) < 1

In [ ]:
matriz = trx.pivot_table(index="categoria", columns="canal", values="margen_bruto", aggfunc="sum")
print("% del margen bruto total, por categoría y canal:")
print((100 * matriz / bruto).round(1).to_string())

tasa = trx.pivot_table(index="categoria", columns="canal", values="margen_bruto", aggfunc="sum") / \
       trx.pivot_table(index="categoria", columns="canal", values="monto_neto", aggfunc="sum")
print("\nMargen bruto % (margen / ventas), por categoría y canal:")
print((100 * tasa).round(1).to_string())
print("\nDescuento promedio por canal:", (trx.groupby("canal").descuento.mean() * 100).round(1).to_dict())

In [ ]:
# Gráfico: % de las ventas contra % del margen neto, por categoría
t = tablas["categoria"].sort_values("% de las ventas")
y = np.arange(len(t))
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(y + 0.2, t["% de las ventas"], height=0.38, color=GRIS, label="% de las ventas")
ax.barh(y - 0.2, t["% del margen neto"], height=0.38, color=AZUL, label="% del margen neto")
for i, (v, m) in enumerate(zip(t["% de las ventas"], t["% del margen neto"])):
    ax.text(v + 0.5, i + 0.2, f"{v:.0f}%", va="center", color=TEXTO_2, fontsize=10)
    ax.text(m + 0.5, i - 0.2, f"{m:.0f}%", va="center", color=TEXTO, fontsize=10)
ax.set_yticks(y, [c.replace("_", " ") for c in t.index])
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")
e = tablas["categoria"].loc["Electrónica"]
ax.set_title(f"Electrónica es el {e['% de las ventas']:.0f}% de las ventas, pero solo el {e['% del margen neto']:.0f}% del margen neto")
ax.legend(loc="lower right", frameon=False)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "a2_categorias_ventas_vs_margen.png", dpi=200)
plt.show()

## 3. Valor del cliente a lo largo del tiempo (CLV)

**Cómo lo calculamos:** CLV = margen neto de un semestre × (1 + r + r² + … + r⁵), donde r es la retención semestral. Es el margen que se espera que deje un cliente en los próximos 3 años (6 semestres) si sigue comprando al mismo ritmo.

- **Margen de un semestre:** margen neto promedio de los clientes del segmento que compraron en ese semestre.
- **Retención semestral (r):** de los clientes que compraron en un semestre, qué porcentaje volvió a comprar en el siguiente.

Usamos semestres y no años porque con 2 años de datos solo hay un paso de un año al siguiente. Con semestres hay tres (2023-S1→S2, 2023-S2→2024-S1 y 2024-S1→S2), y eso permite ver si el CLV cae con el tiempo. Al final comprobamos que la versión anual ordena los segmentos igual.

**Cuándo decimos que un segmento está en declive:** cuando el CLV de 2024 es menor que el de 2023 y la caída se sostiene al remuestrear los clientes 1.000 veces (*bootstrap*). Si el intervalo de confianza del 95% incluye el 0, la diferencia puede ser casualidad y no la llamamos declive.

In [ ]:
# Semestres y clientes activos en cada uno
trx["semestre"] = trx.fecha.dt.year.astype(str) + "-S" + np.where(trx.fecha.dt.month <= 6, "1", "2")
SEMESTRES = sorted(trx.semestre.unique())
margen_sem = trx.groupby(["cliente_id", "semestre"]).margen_neto.sum().unstack(fill_value=0)
activo = trx.groupby(["cliente_id", "semestre"]).size().unstack(fill_value=0) > 0
segmento = cli.set_index("cliente_id").segmento.reindex(activo.index)

HORIZONTE = 6  # semestres = 3 años

def clv(ids, base, siguiente):
    # CLV a 3 años de los clientes activos en el semestre base
    act = activo.loc[ids, base]
    if act.sum() == 0:
        return np.nan, np.nan, np.nan
    m = margen_sem.loc[ids, base][act].mean()
    r = activo.loc[ids, siguiente][act].mean()
    return m * sum(r ** k for k in range(HORIZONTE)), m, r

filas = []
for seg in sorted(segmento.unique()):
    ids = segmento.index[segmento == seg]
    for base, sig in zip(SEMESTRES, SEMESTRES[1:]):
        valor, m, r = clv(ids, base, sig)
        filas.append({"segmento": seg, "periodo": f"{base} → {sig}", "margen por semestre": m,
                      "retención semestral %": 100 * r, "CLV 3 años": valor})
clv_tabla = pd.DataFrame(filas)
print(clv_tabla.pivot(index="segmento", columns="periodo", values="CLV 3 años").round(0).to_string())
print()
print(clv_tabla.pivot(index="segmento", columns="periodo", values="retención semestral %").round(1).to_string())
print()
print(clv_tabla.pivot(index="segmento", columns="periodo", values="margen por semestre").round(1).to_string())

In [ ]:
# ¿Hay declive? Diferencia entre el último periodo y el primero, con intervalo de confianza por bootstrap
rng = np.random.default_rng(42)
primero, ultimo = (SEMESTRES[0], SEMESTRES[1]), (SEMESTRES[2], SEMESTRES[3])
res = []
for seg in sorted(segmento.unique()):
    ids = np.asarray(segmento.index[segmento == seg])
    base = clv(ids, *ultimo)[0] - clv(ids, *primero)[0]
    difs = []
    for _ in range(1000):
        muestra = rng.choice(ids, size=len(ids), replace=True)
        difs.append(clv(muestra, *ultimo)[0] - clv(muestra, *primero)[0])
    lo, hi = np.nanpercentile(difs, [2.5, 97.5])
    inicial = clv(ids, *primero)[0]
    res.append({"segmento": seg, "CLV 2023 (S1→S2)": inicial, "CLV 2024 (S1→S2)": inicial + base,
                "cambio %": 100 * base / inicial, "IC 95% bajo %": 100 * lo / inicial, "IC 95% alto %": 100 * hi / inicial,
                "¿declive?": "sí" if hi < 0 else "no"})
declive = pd.DataFrame(res).set_index("segmento").round(1)
declive

In [ ]:
# Contraste: CLV con el segmento calculado en A1 (último periodo)
rfm = cli.set_index("cliente_id").segmento_rfm.reindex(activo.index)
contraste = {s: clv(rfm.index[rfm == s], *ultimo)[0] for s in sorted(rfm.dropna().unique())}
print("CLV 3 años con el segmento calculado:", {k: round(v) for k, v in contraste.items() if not np.isnan(v)})

# Comprobación con la versión anual (2023 → 2024)
a23 = trx[trx.anio == 2023].groupby("cliente_id").margen_neto.sum()
a24 = set(trx[trx.anio == 2024].cliente_id)
anual = pd.DataFrame({"m": a23, "seg": segmento.reindex(a23.index), "vuelve": a23.index.isin(a24)})
anual = anual.groupby("seg").agg(m=("m", "mean"), r=("vuelve", "mean"))
anual["CLV 3 años (anual)"] = anual.m * (1 + anual.r + anual.r ** 2)
print((anual.assign(r=100 * anual.r)).round(1).to_string())

In [ ]:
# ¿Llegan cada vez menos clientes nuevos, o es que se agotan los clientes del archivo?
primera = trx.groupby("cliente_id").fecha.min().dt.to_period("Q")
trimestres = sorted(primera.unique())
filas = []
pendientes = len(cli)  # clientes del archivo que todavía no han comprado
for q in trimestres:
    nuevos = (primera == q).sum()
    filas.append({"trimestre": str(q), "primeras compras": nuevos, "clientes sin comprar aún": pendientes,
                  "% que compra por primera vez": 100 * nuevos / pendientes})
    pendientes -= nuevos
nuevos_q = pd.DataFrame(filas).set_index("trimestre").round(1)
nuevos_q

In [ ]:
# Gráfico: CLV por segmento con su intervalo
dcl = declive.sort_values("CLV 2024 (S1→S2)")
fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(dcl))
ax.barh(y, dcl["CLV 2024 (S1→S2)"], height=0.55, color=AZUL)
for i, v in enumerate(dcl["CLV 2024 (S1→S2)"]):
    ax.text(v + dcl["CLV 2024 (S1→S2)"].max() * 0.01, i, miles(v), va="center", color=TEXTO, fontsize=10)
ax.set_yticks(y, dcl.index)
ax.xaxis.set_major_formatter(lambda x, _: miles(x))
p, e = declive.loc["Premium", "CLV 2024 (S1→S2)"], declive.loc["Esporádico", "CLV 2024 (S1→S2)"]
ax.set_title(f"Un cliente Premium deja {p / e:.0f} veces más margen en 3 años que uno Esporádico")
ax.set_xlabel("Margen neto esperado por cliente en 3 años (con datos de 2024)")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "a2_clv_segmentos.png", dpi=200)
plt.show()

## 4. Estacionalidad

Para cada categoría y cada canal buscamos el mes pico (el que más vende, en promedio de 2023 y 2024), el mes valle (el que menos vende) y la tendencia (cuánto crecen o bajan las ventas por año).

Después comprobamos si esos picos y valles son reales o casualidad:

- **¿El patrón se repite?** Si un mes es pico por una razón real (Navidad, regreso a clases), debería serlo en los dos años. Lo medimos con la correlación entre el perfil mensual de 2023 y el de 2024: cerca de 1, se repite; cerca de 0 o negativa, no.
- **¿Algún mes vende distinto?** Con la prueba de Kruskal-Wallis comparamos las ventas diarias de los 12 meses. Si p es mayor que 0,05, no hay evidencia de que algún mes venda distinto.
- **¿La tendencia es real?** Si el intervalo de confianza de la tendencia incluye el 0, no podemos decir que las ventas suban o bajen.

In [ ]:
def estacionalidad(col):
    mensual = trx.groupby([col, "mes"]).monto_neto.sum().unstack(0)
    filas = []
    for g in mensual.columns:
        s = mensual[g]
        por_mes = s.groupby(s.index.month).mean()
        indice = 100 * por_mes / por_mes.mean()
        a23 = s[s.index.year == 2023].values
        a24 = s[s.index.year == 2024].values
        t = np.arange(len(s))
        ajuste = stats.linregress(t, s.values)
        pendiente_anual = 100 * ajuste.slope * 12 / s.mean()
        ic = 100 * 1.96 * ajuste.stderr * 12 / s.mean()
        filas.append({col: g, "mes pico": int(indice.idxmax()), "índice pico": indice.max(),
                      "mes valle": int(indice.idxmin()), "índice valle": indice.min(),
                      "¿se repite? (corr. 2023 vs 2024)": np.corrcoef(a23, a24)[0, 1],
                      "tendencia %/año": pendiente_anual, "IC 95% ±": ic})
    return pd.DataFrame(filas).set_index(col).round(2)

print(estacionalidad("categoria").to_string())
print()
print(estacionalidad("canal").to_string())

In [ ]:
diario = trx.groupby("fecha").monto_neto.sum()
grupos = [diario[diario.index.month == m].values for m in range(1, 13)]
h, p = stats.kruskal(*grupos)
print(f"Kruskal-Wallis, ventas diarias por mes (total): H = {h:.1f}, p = {p:.3f}")
indice_total = diario.groupby(diario.index.month).mean()
print("Índice mensual total (100 = mes promedio):", (100 * indice_total / indice_total.mean()).round(0).astype(int).to_dict())

dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia = diario.groupby(diario.index.dayofweek).mean()
print("Venta diaria promedio por día de la semana:", dict(zip(dias, por_dia.round(0).astype(int))))
h, p = stats.kruskal(*[diario[diario.index.dayofweek == k].values for k in range(7)])
print(f"Kruskal-Wallis por día de la semana: p = {p:.3f}")

mensual_total = trx.groupby("mes").monto_neto.sum()
ajuste = stats.linregress(np.arange(24), mensual_total.values)
print(f"Tendencia de las ventas totales: {100 * ajuste.slope * 12 / mensual_total.mean():.1f}% por año (p = {ajuste.pvalue:.2f})")
print("Ventas 2023 vs 2024 por categoría (cambio %):",
      (100 * (trx[trx.anio == 2024].groupby("categoria").monto_neto.sum() / trx[trx.anio == 2023].groupby("categoria").monto_neto.sum() - 1)).round(1).to_dict())
print("Margen bruto 2023 vs 2024 por categoría (cambio %):",
      (100 * (trx[trx.anio == 2024].groupby("categoria").margen_bruto.sum() / trx[trx.anio == 2023].groupby("categoria").margen_bruto.sum() - 1)).round(1).to_dict())

## 5. ¿Los descuentos compensan el margen que regalan?

Un descuento baja el margen de cada venta. Para que valga la pena, tiene que hacer que la gente compre más. Lo comprobamos de cuatro formas:

1. **Por nivel de descuento:** unidades por compra y margen para cada banda de descuento.
2. **Volumen necesario contra volumen real:** si un producto deja un margen g a precio de lista y se le da un descuento d, hay que vender d / (g − d) más unidades para ganar lo mismo. Lo comparamos con lo que de verdad se vendió de más.
3. **Regresión:** el efecto del descuento sobre las unidades por compra, dejando fijos la categoría, el canal y el segmento. Así evitamos confundir el efecto del descuento con el de, por ejemplo, vender más online.
4. **Frecuencia:** si los clientes que reciben más descuento compran más veces, y si el mismo producto vende más en los meses con más descuento.

In [ ]:
trx["banda"] = pd.cut(trx.descuento, [-0.001, 0, 0.05, 0.10, 0.20, 0.35],
                      labels=["sin descuento", "hasta 5%", "5% a 10%", "10% a 20%", "más de 20%"])
bandas = trx.groupby("banda", observed=True).agg(
    compras=("transaccion_id", "size"),
    unidades_por_compra=("cantidad", "mean"),
    descuento_medio=("descuento", "mean"),
    margen_por_compra=("margen_bruto", "mean"),
    ventas=("monto_neto", "sum"),
    margen=("margen_bruto", "sum"),
    regalado=("costo_descuento", "sum"),
)
# Margen que tendría cada venta a precio de lista (sin descuento)
trx["margen_lista"] = 1 - trx.costo_total / (trx.cantidad * trx.precio_unitario)
bandas["margen_lista"] = trx.groupby("banda", observed=True).margen_lista.mean()
bandas["margen %"] = 100 * bandas.margen / bandas.ventas
# Volumen extra necesario para que el margen total no cambie: d / (g - d)
bandas["volumen extra necesario %"] = 100 * bandas.descuento_medio / (bandas.margen_lista - bandas.descuento_medio)
bandas["volumen extra real %"] = 100 * (bandas.unidades_por_compra / bandas.loc["sin descuento", "unidades_por_compra"] - 1)
bandas.drop(columns=["ventas", "margen"]).round(2)

In [ ]:
modelo = smf.ols("cantidad ~ descuento + C(categoria) + C(canal) + C(segmento)", data=trx).fit()
coef, (lo, hi) = modelo.params["descuento"], modelo.conf_int().loc["descuento"]
print(f"Efecto del descuento sobre las unidades por compra (controlando categoría, canal y segmento):")
print(f"   +10 puntos de descuento → {coef * 0.10:+.3f} unidades (IC 95%: {lo * 0.10:+.3f} a {hi * 0.10:+.3f}), p = {modelo.pvalues['descuento']:.2f}")

# ¿Los clientes que reciben más descuento compran más veces?
por_cliente = trx.groupby(["cliente_id", "segmento"]).agg(compras=("transaccion_id", "size"), descuento=("descuento", "mean")).reset_index()
for seg, g in por_cliente.groupby("segmento"):
    if g.compras.nunique() > 1:
        rho, p = stats.spearmanr(g.descuento, g.compras)
        print(f"   {seg:11s} correlación descuento medio vs número de compras: {rho:+.3f} (p = {p:.2f})")

# ¿Los meses con más descuento en un producto venden más unidades?
pm = trx.groupby(["producto_id", "mes"]).agg(unidades=("cantidad", "sum"), descuento=("descuento", "mean")).reset_index()
pm["unidades_rel"] = pm.unidades / pm.groupby("producto_id").unidades.transform("mean")
pm["descuento_rel"] = pm.descuento - pm.groupby("producto_id").descuento.transform("mean")
rho, p = stats.spearmanr(pm.descuento_rel, pm.unidades_rel)
print(f"Mismo producto, meses con más descuento vs unidades vendidas: correlación {rho:+.3f} (p = {p:.2f})")

In [ ]:
print(f"Descuentos regalados en total: {regalado:,.0f}")
por_canal = trx.groupby("canal").costo_descuento.sum()
por_cat = trx.groupby("categoria").costo_descuento.sum()
print("Por canal (% del total):", (100 * por_canal / regalado).round(1).to_dict())
print("   % de las ventas de cada canal:", (100 * trx.groupby("canal").monto_neto.sum() / ventas).round(1).to_dict())
print("Por categoría (% del total):", (100 * por_cat / regalado).round(1).to_dict())
bc = trx[trx.bajo_costo]
print(f"Ventas bajo costo: {len(bc)}, pérdida {-bc.margen_bruto.sum():,.0f}; por canal:", bc.canal.value_counts().to_dict())
print("Descuento promedio por canal y año (%):")
print((100 * trx.groupby(["canal", "anio"]).descuento.mean().unstack()).round(2).to_string())

In [ ]:
# Gráfico: margen por compra y unidades por compra según el descuento (dos paneles, sin doble eje)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
x = np.arange(len(bandas))
etiquetas = [b.replace(" a ", "–") for b in bandas.index.astype(str)]
a1.bar(x, bandas["margen %"], width=0.6, color=AZUL)
for i, v in enumerate(bandas["margen %"]):
    a1.text(i, v + 0.6, f"{v:.0f}%", ha="center", color=TEXTO, fontsize=10)
a1.set_xticks(x, etiquetas, rotation=20)
a1.set_title("Margen bruto %", fontsize=11)
a1.set_ylim(0, 34)
a1.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
a2.bar(x, bandas["unidades_por_compra"], width=0.6, color=GRIS)
for i, v in enumerate(bandas["unidades_por_compra"]):
    a2.text(i, v + 0.03, f"{v:.2f}".replace(".", ","), ha="center", color=TEXTO, fontsize=10)
a2.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}".replace(".", ","))
a2.set_xticks(x, etiquetas, rotation=20)
a2.set_ylim(0, 2.2)
a2.set_title("Unidades por compra", fontsize=11)
for a in (a1, a2):
    a.grid(axis="x", visible=False)
fig.suptitle(f"Con más descuento el margen cae de {bandas['margen %'].iloc[0]:.0f}% a {bandas['margen %'].iloc[-1]:.0f}%, "
             "pero la gente compra las mismas unidades", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(FIG / "a2_descuentos.png", dpi=200)
plt.show()

## 6. Resumen de A2

**Quién aporta el margen**
- **Categorías:** Hogar y Deco es la que más margen bruto aporta (31%). Electrónica es el 44% de las ventas, pero solo el 24% del margen bruto y el 16% del margen neto: tiene el margen más bajo (13,7%) y un 18% de sus ventas se devuelve. La que menos aporta es Alimentos y Bebidas (2%).
- **Canales:** la tienda física aporta el 47,5% del margen bruto; online, el 34,6%; la app, el 17,9%. Online tiene el margen más bajo (22,8%) porque da el doble de descuento (7,5% contra 3,7% en tienda y app).
- **Segmentos:** los Frecuentes aportan la mitad del margen bruto (49,5%) y los Premium, casi un tercio (31,2%). Los Inactivos (1,8%) y los Nuevos (1,9%) casi no aportan.
- **Devoluciones:** se llevan el 27,5% del margen bruto. **Descuentos:** se llevan el 18,3% del margen que habría sin ellos.

**Valor del cliente (CLV a 3 años)**
- Un cliente Premium deja 1.828 de margen neto en 3 años; uno Frecuente, 851; uno Esporádico, 145; uno Nuevo, 90; uno Inactivo, 57.
- **Ningún segmento está en declive.** Los cambios entre 2023 y 2024 van de −16% a +15%, pero en todos los segmentos el intervalo de confianza incluye el 0. La retención de cada segmento es casi idéntica en los tres periodos.
- Las primeras compras bajan de 3.499 (inicio de 2023) a 218 (fin de 2024), pero no porque lleguen menos clientes: el archivo tiene 8.000 clientes fijos y se van quedando sin clientes que no hayan comprado. De los que aún no habían comprado, entre el 25% y el 36% compra por primera vez cada trimestre, sin tendencia a la baja.

**Estacionalidad**
- Cada categoría y canal tiene un mes pico y un mes valle (ver la tabla de la sección 4), con diferencias de ±10% respecto a un mes normal.
- Pero **esos picos y valles no se repiten** de un año al otro (correlación entre −0,35 y 0,33), ningún mes vende distinto (p = 0,63) y ningún día de la semana tampoco (p = 0,99). No hay estacionalidad: las diferencias entre meses son ruido.
- Tampoco hay tendencia: las ventas totales crecen 0,7% al año, y no es significativo (p = 0,71). Electrónica es la única categoría cuyo margen bruto baja de 2023 a 2024 (−2,6%). Es una caída pequeña, pero va en la misma dirección que el resto de sus problemas.

**Descuentos**
- **No compensan.** Con más descuento el margen cae de 29% a 7%, pero las unidades por compra son las mismas (1,78). Con un descuento de entre 10% y 20% habría que vender un 75% más de unidades para ganar lo mismo; con más de 20%, el triple. Y se vende igual.
- La regresión lo confirma: 10 puntos más de descuento suman +0,004 unidades por compra (IC 95%: −0,012 a +0,020), es decir, nada.
- El margen de lista es el mismo en todas las bandas de descuento (34%): los descuentos no se dan a productos más caros o más baratos. Eso hace más creíble que la falta de efecto sea real y no una casualidad de a qué productos se les dio descuento.
- Los clientes que reciben más descuento compran apenas un poco más veces (correlación de 0,05 a 0,09, solo significativa en Esporádico y Nuevo). Es una asociación muy débil y no prueba que el descuento sea la causa.
- **Online** concentra el 56% de lo regalado en descuentos, con solo el 38% de las ventas. **Electrónica**, el 45%. Las 711 ventas bajo costo de A1 (441 de ellas online) suman 30.181 de pérdida.

**Títulos de gráfico candidatos para A5**
- "Electrónica es el 44% de las ventas, pero solo el 16% del margen neto" (`figures/a2_categorias_ventas_vs_margen.png`)
- "Un cliente Premium deja 13 veces más margen en 3 años que uno Esporádico" (`figures/a2_clv_segmentos.png`)
- "Con más descuento el margen cae de 29% a 7%, pero la gente compra las mismas unidades" (`figures/a2_descuentos.png`)